# Aperture2D

Two plates mirrored about a beam axis: a slit, or the 2D cut through a round aperture.

`Aperture2D(axis, position, vertices)` takes the profile of one plate as `(axial, radial)`
vertices, the coordinate along the beam and the distance from the centre line at
`position`; the second plate is its mirror image. `Aperture2D.slit` builds the common
rectangular plates.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from simgeometry import Aperture2D
from simgeometry.crossings import crossings

slit = Aperture2D.slit(axial_position=0.3, half_width=0.1, thickness=0.05, length=0.4)
# a knife edge with a narrower bore: thin at the bore, thicker outside
knife = Aperture2D(
    axis=0,
    position=0.0,
    vertices=[(0.8, 0.05), (0.84, 0.05), (0.95, 0.5), (0.7, 0.5)],
)
print(
    "on the axis:", slit.contains((0.3, 0.0)), " in a plate:", slit.contains((0.3, 0.2))
)
assert not slit.contains((0.3, 0.0)) and slit.contains((0.3, -0.2))

## A beam through both apertures

Steps along the beam axis (x) at different heights: the slit stops those with $|y| > 0.1$,
the knife edge those with $|y| > 0.05$ that got through the slit.

In [ ]:
y = np.linspace(-0.425, 0.425, 18)
start = np.stack((np.zeros_like(y), y))
end = np.stack((np.ones_like(y), y))
first = crossings(slit, start, end)
end_after_slit = np.where(first.hit, first.point, end)
second = crossings(knife, start, end_after_slit)
stopped = np.where(second.hit, second.point, end_after_slit)
assert first.hit.sum() == np.sum(np.abs(y) > 0.1)
assert second.hit.sum() == np.sum((np.abs(y) > 0.05) & (np.abs(y) < 0.1))

fig, ax = plt.subplots(figsize=(6, 3.6), layout="constrained")
for shape, color in ((slit, "tab:blue"), (knife, "tab:green")):
    for loop in shape.outlines():
        ax.fill(*loop, color=color, alpha=0.5)
for i in range(len(y)):
    ax.plot([start[0, i], stopped[0, i]], [y[i], y[i]], color="tab:orange", lw=1)
ax.axhline(0.0, color="0.5", lw=0.8, ls="--")
ax.set(xlabel="x (beam axis)", ylabel="y", title="A slit and a knife-edge aperture")
ax.set_aspect("equal")
plt.show()